# QLoRA Summarization — Colab Training Pipeline

In [1]:
GITHUB_REPO = "https://github.com/ducnd58233/language-engineer.git"
GITHUB_TOKEN = ""
HF_TOKEN = ""
HF_REPO_ID = "ducnd58233/qwen2.5-3b-qlora-summarization"
EPOCHS = 1
N_EVAL_SAMPLES = 100

In [2]:
import os
import subprocess

try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN") or GITHUB_TOKEN
except Exception:
    pass

repo_url = GITHUB_REPO.replace("https://", f"https://{GITHUB_TOKEN}@") if GITHUB_TOKEN else GITHUB_REPO

if not os.path.exists("language-engineer"):
    subprocess.run(["git", "clone", repo_url, "language-engineer"], check=True)
else:
    subprocess.run(["git", "-C", "language-engineer", "pull"], check=True)

In [3]:
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121
!pip install -q peft trl transformers accelerate bitsandbytes datasets \
    rouge-score sacrebleu bert-score python-dotenv pyyaml tqdm

In [4]:
%cd language-engineer

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN") or HF_TOKEN
except Exception:
    pass

with open(".env", "w") as f:
    f.write(f"HF_TOKEN={HF_TOKEN}\n")
    f.write(f"HF_REPO_ID={HF_REPO_ID}\n")

/content/language-engineer


## 2. (Optional) Mount Google Drive

Uncomment to persist checkpoints and datasets across Colab sessions.
Without this, all data is lost when the runtime disconnects.

In [5]:
# from google.colab import drive
# drive.mount("/content/drive")
#
# os.makedirs("/content/drive/MyDrive/lang-eng/runs", exist_ok=True)
# os.makedirs("/content/drive/MyDrive/lang-eng/datasets", exist_ok=True)
# if not os.path.exists("runs"):
#     os.symlink("/content/drive/MyDrive/lang-eng/runs", "runs")
# if not os.path.exists("datasets"):
#     os.symlink("/content/drive/MyDrive/lang-eng/datasets", "datasets")

## 3. Prepare Datasets

In [6]:
!python scripts/prepare_datasets.py

Creating parquet from Arrow format: 100% 288/288 [00:14<00:00, 19.55ba/s]
Creating parquet from Arrow format: 100% 205/205 [00:07<00:00, 29.17ba/s]
Creating parquet from Arrow format: 100% 14/14 [00:00<00:00, 22.94ba/s]
Creating parquet from Arrow format: 100% 12/12 [00:00<00:00, 35.08ba/s]
Creating parquet from Arrow format: 100% 12/12 [00:00<00:00, 16.10ba/s]
Creating parquet from Arrow format: 100% 12/12 [00:00<00:00, 35.95ba/s]


In [7]:
!python scripts/process_datasets.py

Using num_proc=2
Datasets found: ['cnn', 'xsum']
Setting num_proc from 2 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Generating train split: 287113 examples [00:14, 19819.42 examples/s]
Setting num_proc from 2 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Generating train split: 204017 examples [00:05, 39746.04 examples/s]
Setting num_proc from 2 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Generating train split: 13368 examples [00:00, 26382.11 examples/s]
Setting num_proc from 2 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Generating train split: 11327 examples [00:00, 69521.05 examples/s]
Setting num_proc from 2 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Generating train split: 11490 examples [00:00, 40973.13 examples/s]
Setting num_proc from 2 back to 1 for the train s

## 4. Train

In [ ]:
!python scripts/train.py --epochs {EPOCHS}

Loading tokenizer: Qwen/Qwen2.5-3B-Instruct
Loading model: Qwen/Qwen2.5-3B-Instruct (4-bit NF4)
Loading weights: 100% 434/434 [00:25<00:00, 17.22it/s, Materializing param=model.norm.weight]                               
trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607
Loading datasets...
Train shards: 5, val shards: 1
Training: 449004 rows x 1 epochs = 56126 steps
[RANK 0] Padding-free training is enabled, but the attention implementation is not set to a supported flash attention variant. Padding-free training flattens batches into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn2, kernels-community/flash-attn3, kernels-community/vllm-flash-attn3. Using other implementations may lead to unexpected behavior. To ensure compatibility, set `attn_implementation` in the model configuration to one of these supported options or verify that your attenti

## 5. Evaluate

Compares base model (zero-shot) vs. fine-tuned adapter on the test set.
Results saved to `results/eval_results.json`.

In [ ]:
!python scripts/evaluate.py --n-samples {N_EVAL_SAMPLES}

## 6. Upload to HuggingFace Hub

In [ ]:
!python scripts/hub.py upload \
    --adapter runs/Qwen2.5-3B-Instruct_qlora/final \
    --repo {HF_REPO_ID}

## 7. (Optional) Download from HuggingFace Hub

Pull a previously uploaded adapter back to local disk (e.g. to resume eval on a new runtime).

In [ ]:
!python scripts/hub.py download \
    --repo {HF_REPO_ID} \
    --output runs/downloaded